# MV-Kubric WebDataset pilot

This notebook is a thin Modal Notebook driver. Attach the existing `jeet-mvtracker-data-v2` Volume at `/mnt/mvtracker-data`, select one T4 with 16 CPUs and 32 GB RAM, and run the CPU conversion function below. The function is tagged `owner=jeet`, `project=mvtracker`, `purpose=profiling` and logs progress to W&B.

The pilot converts a small explicit scene list into four-scene uncompressed TAR shards with NVIDIA `wds2idx` indices. It does not delete native data or launch training.

In [ ]:
from pathlib import Path
import json

DATA_ROOT = Path('/mnt/mvtracker-data')
SCENE_ROOT = DATA_ROOT / 'datasets/kubric-multiview/train'
OUTPUT_ROOT = DATA_ROOT / 'datasets/kubric-multiview-webdataset/v1/train'
pilot_scenes = tuple(str(scene) for scene in range(900, 932))
print(f'{len(pilot_scenes)} scenes: {pilot_scenes[0]}..{pilot_scenes[-1]}')

## Convert the pilot

The `%modal` invocation is intentionally explicit. Use `shard_workers=8` only when eight CPU slots are available; otherwise leave it at one.

In [ ]:
# In Modal Notebook, invoke the deployed function with the repository checkout.
# %modal run tools/modal_mvkubric_webdataset.py::convert \
#   --scene-root /mnt/mvtracker-data/datasets/kubric-multiview/train \
#   --output-root /mnt/mvtracker-data/datasets/kubric-multiview-webdataset/v1/train \
#   --scenes 900,901,902,903,904,905,906,907,908,909,910,911,912,913,914,915,916,917,918,919,920,921,922,923,924,925,926,927,928,929,930,931 \
#   --scenes-per-shard 4 --shard-workers 1 --read-workers 16
print('Run the commented %modal command after attaching the Volume and selecting a T4.')

In [ ]:
manifest = OUTPUT_ROOT / 'manifest.json'
if manifest.exists():
    report = json.loads(manifest.read_text())
    print(f"converted scenes={len(report['scene_ids'])} shards={len(report['shards'])}")
    for shard in report['shards']:
        print(shard['name'], shard['scene_ids'], shard['status'])
else:
    print('No manifest yet; run the conversion cell first.')